In [2]:
import torch
def matrix_log_eig(A: torch.Tensor) -> torch.Tensor:
    """Matrix log via eigendecomposition. Works for any diagonalizable matrix."""
    eigenvalues, V = torch.linalg.eig(A)  # complex
    log_diag = torch.diag_embed(torch.log(eigenvalues))
    result = V @ log_diag @ torch.linalg.inv(V)
    return result.real  # discard numerical imaginary noise

def geodesic_gl3(L0: torch.Tensor, L1: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """
    Geodesic on GL+(3) with left-invariant metric.
    L_t = L0 @ expm(t * logm(L0^{-1} @ L1))
    
    Args:
        L0: (batch, 3, 3) source matrices
        L1: (batch, 3, 3) target matrices
        t:  (batch, 1, 1) or scalar, time in [0, 1]
    Returns:
        Lt: (batch, 3, 3)
    """
    L0_inv = torch.linalg.inv(L0)
    V = matrix_log_eig(L0_inv @ L1)
    Lt = L0 @ torch.linalg.matrix_exp(t * V)
    return Lt


def velocity_gl3(L0: torch.Tensor, L1: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """
    Conditional vector field: u_t = L_t @ V where V = logm(L0^{-1} @ L1)
    
    Returns:
        u_Lt: (batch, 3, 3) velocity in ambient R^{3x3}
    """
    L0_inv = torch.linalg.inv(L0)
    V = matrix_log_eig(L0_inv @ L1)
    Lt = L0 @ torch.linalg.matrix_exp(t * V)
    u_Lt = Lt @ V
    return u_Lt


# Example
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch = 1024

L0 = torch.randn(batch, 3, 3, device=device)
L1 = torch.randn(batch, 3, 3, device=device)

# Make sure they're in GL+(3): force positive determinant
L0 = L0 * torch.sign(torch.linalg.det(L0)).unsqueeze(-1).unsqueeze(-1)
L1 = L1 * torch.sign(torch.linalg.det(L1)).unsqueeze(-1).unsqueeze(-1)

t = torch.rand(batch, 1, 1, device=device)

Lt = geodesic_gl3(L0, L1, t)
u = velocity_gl3(L0, L1, t)

print(f"Device: {device}")
print(f"Lt shape: {Lt.shape}")
print(f"u shape: {u.shape}")
print(f"All det(Lt) > 0: {(torch.linalg.det(Lt) > 0).all().item()}")

# Numerical verification: compare u with finite difference
eps = 1e-5
Lt_fwd = geodesic_gl3(L0, L1, t + eps)
u_numerical = (Lt_fwd - Lt) / eps
print(f"Max velocity error vs finite diff: {(u - u_numerical).abs().max().item():.2e}")

Device: cuda
Lt shape: torch.Size([1024, 3, 3])
u shape: torch.Size([1024, 3, 3])
All det(Lt) > 0: True
Max velocity error vs finite diff: 2.93e+02
